# PathMNIST FL: plaintext FedAvg vs CKKS (Microsoft SEAL) with a third-party decryptor

Paired experiments matching the best baseline pair from the 10-experiment report:

| Run | Distribution | Aggregation | Config (= T09 / T10) |
|---|---|---|---|
| 1 | IID | plaintext | 5 clients · 20 rounds · 5 local epochs · batch 128 · SGD lr 0.01, momentum 0.9 · seed 42 |
| 2 | IID | CKKS | same |
| 3 | Non-IID (Dirichlet α=0.5) | plaintext | same |
| 4 | Non-IID (Dirichlet α=0.5) | CKKS | same |

Plus a HE-only cost benchmark (client count 2/3/5/10, CKKS parameter variants, RAM-disk serialization).

* Uses the repo's own `run_federated_training`, `CKKSStrategy`, `CKKSAggregator` (public key only) and `TrustedDecryptor` (secret key) **unchanged**; the runner only adds measurement hooks (per-round CPU time, a plaintext *shadow* FedAvg on the same client models to measure the real CKKS error, peak RAM).
* Each plaintext/CKKS twin uses the same seed, partition, model init and client order, with deterministic GPU kernels, so differences come from encrypted aggregation only.
* **Runtime:** ≈25 min per run on a T4 (≈1h45 total). Runs resume: if Colab disconnects, re-run the cells and finished runs are skipped. Mount Drive (cell 2) to keep results across disconnects.
* **Output:** `ckks_vs_plaintext_report.html` (self-contained) + `results.zip`.

Runtime → Change runtime type → **T4 GPU**, then Run all.

In [ ]:
!nvidia-smi -L || echo 'No GPU: runs will work on CPU but take ~1h each'
!python --version

In [ ]:
# Optional: keep results on Google Drive so a disconnect doesn't lose finished runs
USE_DRIVE = True
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/FYP_ckks_experiments'
else:
    OUT = '/content/FYP_ckks_experiments'
os.makedirs(OUT, exist_ok=True)
print('Results go to', OUT)

In [ ]:
%cd /content
!test -d FYP || git clone -q https://github.com/Majdah05/FYP.git
%cd /content/FYP
!git log --oneline -1
!pip install -q -r requirements.txt -r requirements-he.txt
!python -c "import tenseal, torch; from tenseal import sealapi; print('tenseal', tenseal.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"
!mkdir -p /content/ckks_exp

### Experiment scripts (written to `/content/ckks_exp/`)

In [ ]:
%%writefile /content/ckks_exp/run_fl.py
"""Run one PathMNIST FedAvg experiment (plaintext or CKKS) with the repository's
own training loop, adding measurement-only instrumentation:

* per-round process CPU time and wall time
* plaintext aggregation time (plaintext mode)
* in CKKS mode, a shadow plaintext fedavg_aggregate on the SAME client states,
  giving the real per-round CKKS aggregation error on trained weights
  (measurement only; the shadow result is never used by training)
* peak resident memory

Data: PathMNIST is read from a local pathmnist.npz (download disabled). Images
are normalised once into memory with exactly the repository's ToTensor +
Normalize arithmetic (verified bit-exact at start-up), to avoid per-image PIL
overhead on CPU. Partitioning, client loaders (shuffle=False as in the repo),
model, optimiser and evaluation are the repository's code.
"""
import argparse, copy, json, resource, sys, time
from pathlib import Path


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--repo', required=True)
    ap.add_argument('--data_dir', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--mode', choices=['plaintext', 'ckks'], required=True)
    ap.add_argument('--distribution', choices=['iid', 'non_iid'], required=True)
    ap.add_argument('--num_clients', type=int, default=5)
    ap.add_argument('--rounds', type=int, default=20)
    ap.add_argument('--local_epochs', type=int, default=5)
    ap.add_argument('--batch_size', type=int, default=128)
    ap.add_argument('--lr', type=float, default=0.01)
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--threads', type=int, default=1)
    ap.add_argument('--train_limit', type=int, default=0, help='debug only')
    ap.add_argument('--device', choices=['cpu', 'cuda', 'auto'], default='cpu')
    ap.add_argument('--download', action='store_true', help='let MedMNIST download pathmnist.npz into data_dir')
    a = ap.parse_args()
    sys.path.insert(0, str(Path(a.repo).resolve()))

    import numpy as np, torch
    torch.set_num_threads(a.threads)
    # deterministic kernels so the plaintext/CKKS twins differ only by aggregation numerics
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    device = ('cuda' if torch.cuda.is_available() else 'cpu') if a.device == 'auto' else a.device
    import medmnist
    from torch.utils.data import DataLoader
    from torchvision import transforms
    from main import PathMNISTSubsetFactory, set_seed, evaluate_federated_test
    from federated_learning.config import ExperimentConfig
    from federated_learning.federated import server as server_mod
    from federated_learning.federated import strategies as strat_mod
    from federated_learning.federated.aggregation import fedavg_aggregate
    from federated_learning.federated.server import initialize_global_model, run_federated_training
    from federated_learning.utils.logging import save_json, save_metrics_csv

    out = Path(a.out); out.mkdir(parents=True, exist_ok=False)
    mean = [0.740545, 0.532982, 0.705829]; std = [0.123682, 0.176763, 0.124431]
    tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=mean, std=std)])

    class MemDS(torch.utils.data.Dataset):
        def __init__(self, split):
            Path(a.data_dir).mkdir(parents=True, exist_ok=True)
            raw = medmnist.PathMNIST(root=a.data_dir, split=split, transform=tf, download=a.download)
            n = len(raw)
            m = torch.tensor(mean).view(1, 3, 1, 1); s = torch.tensor(std).view(1, 3, 1, 1)
            self.x = torch.empty((n, 3, 28, 28), dtype=torch.float32)
            for start in range(0, n, 8192):   # chunked to bound peak memory
                chunk = torch.from_numpy(raw.imgs[start:start + 8192]).permute(0, 3, 1, 2).float().div(255.0)
                self.x[start:start + 8192] = chunk.sub_(m).div_(s)
            for i in (0, 1, n // 2, n - 1):
                if not torch.equal(raw[i][0], self.x[i]):
                    raise RuntimeError(f'in-memory preprocessing differs from repo transform at {split}[{i}]')
            self.labels = raw.labels.copy()
            self.y = torch.from_numpy(raw.labels.astype(np.int64))
            del raw
        def __len__(self): return len(self.y)
        def __getitem__(self, i): return self.x[i], self.y[i]

    train, val, test = MemDS('train'), MemDS('val'), MemDS('test')
    val_loader = DataLoader(val, batch_size=a.batch_size, shuffle=False)
    test_loader = DataLoader(test, batch_size=a.batch_size, shuffle=False)
    idx = list(range(a.train_limit)) if a.train_limit else None
    factory = PathMNISTSubsetFactory(train, batch_size=a.batch_size, seed=a.seed, sample_indices=idx)

    cfg = ExperimentConfig(num_clients=a.num_clients, rounds=a.rounds, local_epochs=a.local_epochs,
                           batch_size=a.batch_size, lr=a.lr, momentum=0.9, optimizer='sgd',
                           distribution=a.distribution, dirichlet_alpha=0.5, client_fraction=1.0,
                           model='cnn', seed=a.seed, device=device, output_dir=str(out),
                           save_checkpoints=False, aggregation=a.mode)
    cfg.validate()

    # ---------------- measurement-only instrumentation ----------------
    extra = {}            # round -> dict of extra metrics
    stash = {}            # (round, client) -> plaintext state copy (CKKS shadow only)
    cpu_mark = {'t': None, 'r': None}
    orig_create = strat_mod.create_aggregation_strategy

    def create(config, initial_state):
        st = orig_create(config, initial_state)
        orig_begin, orig_agg = st.begin_round, st.aggregate

        def begin_round(round_id, participants):
            cpu_mark['t'] = time.process_time(); cpu_mark['r'] = round_id
            extra.setdefault(round_id, {})
            return orig_begin(round_id, participants)

        def aggregate(updates, counts, client_metrics):
            rid = cpu_mark['r']
            t, c = time.perf_counter(), time.process_time()
            state, m = orig_agg(updates, counts, client_metrics)
            e = extra[rid]
            e['aggregation_call_wall_seconds'] = time.perf_counter() - t   # server + TTP (CKKS) / fedavg (plain)
            e['aggregation_call_cpu_seconds'] = time.process_time() - c
            if config.aggregation == 'ckks':
                shadow = fedavg_aggregate([stash.pop((rid, u.client_id)) for u in updates], counts)
                ref = np.concatenate([v.double().numpy().ravel() for v in shadow.values() if v.is_floating_point()])
                got = np.concatenate([v.double().numpy().ravel() for v in state.values() if v.is_floating_point()])
                err = np.abs(ref - got)
                e['ckks_vs_plaintext_max_abs_err'] = float(err.max())
                e['ckks_vs_plaintext_rmse'] = float(np.sqrt((err ** 2).mean()))
                e['max_abs_weight'] = float(np.abs(ref).max())
            return state, m

        st.begin_round, st.aggregate = begin_round, aggregate
        if st.client_encoder is not None:
            enc = st.client_encoder
            orig_prepare = enc.prepare

            def prepare(state, sample_count, client_id, round_id):
                stash[(round_id, client_id)] = {k: v.detach().clone() for k, v in state.items()}
                c = time.process_time()
                r = orig_prepare(state, sample_count, client_id, round_id)
                extra[round_id]['client_encrypt_cpu_seconds'] = extra[round_id].get('client_encrypt_cpu_seconds', 0.0) + time.process_time() - c
                return r
            enc.prepare = prepare
        return st

    server_mod.create_aggregation_strategy = create
    orig_eval = server_mod.evaluate_model

    def eval_hook(model, loader, device='cpu', criterion=None):
        # called once per round right after aggregation: close the round's CPU window
        res = orig_eval(model, loader, device=device, criterion=criterion)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        if cpu_mark['t'] is not None:
            extra[cpu_mark['r']]['round_cpu_seconds'] = time.process_time() - cpu_mark['t']
        return res
    server_mod.evaluate_model = eval_hook
    # ------------------------------------------------------------------

    set_seed(a.seed)
    model = initialize_global_model('pathmnist', model_name='cnn', num_classes=9)
    t0, c0 = time.perf_counter(), time.process_time()
    model, history = run_federated_training(factory, val_loader, cfg, output_dir=str(out), global_model=model)
    fl_wall, fl_cpu = time.perf_counter() - t0, time.process_time() - c0
    for h in history:
        h.update(extra.get(h['round'], {}))
    test_metrics = evaluate_federated_test(model, test_loader, cfg, out)
    save_json({'config': cfg.__dict__, 'history': history, 'test_metrics': test_metrics,
               'federated_wall_seconds': fl_wall, 'federated_cpu_seconds': fl_cpu,
               'peak_rss_mb': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024,
               'torch_threads': torch.get_num_threads(), 'device': device,
               'gpu': torch.cuda.get_device_name(0) if device == 'cuda' else None},
              out / 'federated_history.json')
    save_metrics_csv(history, out / 'federated_history.csv')
    torch.save(model.state_dict(), out / 'final_model.pt')
    print('DONE', a.mode, a.distribution, 'test_acc', test_metrics['test_accuracy'], 'wall', fl_wall, flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/ckks_exp/he_cost_bench.py
"""HE-only cost benchmark on the real PathMNIST CNN schema (no dataset needed).

Uses the repository's own CKKSStrategy (client encoder -> public aggregator ->
trusted third-party decryptor) exactly as training does, and measures for one
FedAvg round: wall time, CPU time, per-phase timings, ciphertext bytes and
aggregation error against the plaintext fedavg_aggregate on the same states.

Usage (from repo root):
  python -I /path/he_cost_bench.py --repo . --out OUT.json --clients 2 3 5 10 \
      --params "8192:60,40,40,60:40" --repeats 3
"""
import argparse, copy, json, os, resource, sys, time
from pathlib import Path


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--repo', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--clients', type=int, nargs='+', default=[5])
    ap.add_argument('--params', nargs='+', default=['8192:60,40,40,60:40'])
    ap.add_argument('--repeats', type=int, default=3)
    ap.add_argument('--samples_per_client', type=int, default=17999)
    ap.add_argument('--label', default='')
    a = ap.parse_args()
    sys.path.insert(0, str(Path(a.repo).resolve()))
    import numpy as np, torch
    torch.set_num_threads(1)
    from federated_learning.config import ExperimentConfig
    from federated_learning.federated.aggregation import fedavg_aggregate
    from federated_learning.federated.strategies import create_aggregation_strategy
    from federated_learning.models.cnn import PathMNISTCNN

    torch.manual_seed(0)
    template = PathMNISTCNN(9).state_dict()
    rng = np.random.default_rng(0)
    records = []
    rid = 0
    for p in a.params:
        n, bits, scale = p.split(':')
        cfg = ExperimentConfig(aggregation='ckks', ckks_poly_modulus_degree=int(n),
                               ckks_coeff_mod_bit_sizes=[int(b) for b in bits.split(',')],
                               ckks_global_scale_power=int(scale))
        t0 = time.perf_counter()
        strat = create_aggregation_strategy(cfg, template)
        init_s = time.perf_counter() - t0
        for k in a.clients:
            for rep in range(a.repeats):
                rid += 1
                states = []
                for _ in range(k):
                    s = copy.deepcopy(template)
                    for name, v in s.items():
                        if v.is_floating_point():
                            # realistic magnitudes: trained CNN weights are mostly |w|<0.5
                            s[name] = (v + torch.from_numpy(rng.normal(0, 0.05, tuple(v.shape))).float())
                    states.append(s)
                counts = [a.samples_per_client + i for i in range(k)]
                t = time.perf_counter(); c = time.process_time()
                plain = fedavg_aggregate(states, counts)
                plain_wall, plain_cpu = time.perf_counter() - t, time.process_time() - c
                strat.begin_round(rid, tuple(enumerate(counts)))
                t = time.perf_counter(); c = time.process_time()
                enc_wall_clients, enc_cpu_clients, packs = [], [], []
                for i, (s, cnt) in enumerate(zip(states, counts)):
                    t1 = time.perf_counter(); c1 = time.process_time()
                    packs.append(strat.client_encoder.prepare(s, cnt, i, rid))
                    enc_wall_clients.append(time.perf_counter() - t1); enc_cpu_clients.append(time.process_time() - c1)
                t2 = time.perf_counter(); c2 = time.process_time()
                agg_state, m = strat.aggregate([x[0] for x in packs], counts, [x[1] for x in packs])
                server_ttp_wall, server_ttp_cpu = time.perf_counter() - t2, time.process_time() - c2
                wall, cpu = time.perf_counter() - t, time.process_time() - c
                ref = np.concatenate([v.double().numpy().ravel() for v in plain.values() if v.is_floating_point()])
                got = np.concatenate([v.double().numpy().ravel() for v in agg_state.values() if v.is_floating_point()])
                err = np.abs(ref - got)
                rec = dict(label=a.label, params=p, clients=k, repeat=rep + 1,
                           tmpdir=os.environ.get('TMPDIR', '/tmp(default)'),
                           context_init_seconds=init_s,
                           plaintext_agg_wall_s=plain_wall, plaintext_agg_cpu_s=plain_cpu,
                           ckks_round_wall_s=wall, ckks_round_cpu_s=cpu,
                           per_client_encrypt_wall_s=float(np.mean(enc_wall_clients)),
                           per_client_encrypt_cpu_s=float(np.mean(enc_cpu_clients)),
                           server_plus_ttp_wall_s=server_ttp_wall, server_plus_ttp_cpu_s=server_ttp_cpu,
                           max_abs_err=float(err.max()), rmse=float(np.sqrt((err ** 2).mean())),
                           plaintext_upload_bytes=k * sum(v.numel() * v.element_size() for v in template.values() if v.is_floating_point()),
                           peak_rss_mb=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024,
                           **{kk: vv for kk, vv in m.items() if not isinstance(vv, str)})
                records.append(rec)
                print(json.dumps({x: rec[x] for x in ('params', 'clients', 'repeat', 'ckks_round_wall_s', 'he_serialization_time_seconds',
                                                      'he_ciphertext_upload_bytes', 'max_abs_err')}), flush=True)
    Path(a.out).write_text(json.dumps(records, indent=1))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/ckks_exp/analyze.py
"""Summarise paired plaintext vs CKKS runs + HE benchmarks into one JSON."""
import json, sys
from pathlib import Path
import numpy as np

root = Path(sys.argv[1])          # dir containing iid_plaintext, iid_ckks, non_iid_plaintext, non_iid_ckks
bench = Path(sys.argv[2])         # dir with he_*.json
out = Path(sys.argv[3])

PT_MODEL_BYTES = 241993 * 4


def load(name):
    p = root / name / 'federated_history.json'
    return json.loads(p.read_text()) if p.exists() else None


summary = {'runs': {}, 'pairs': {}}
for dist in ('iid', 'non_iid'):
    pair = {}
    for mode in ('plaintext', 'ckks'):
        d = load(f'{dist}_{mode}')
        if d is None:
            continue
        h, t = d['history'], d['test_metrics']
        k = len(h[0]['selected_client_ids'])
        r = {
            'rounds_completed': len(h),
            'test_accuracy': t['test_accuracy'], 'test_macro_f1': t['macro_f1'], 'test_auroc': t['auroc'],
            'test_loss': t['test_loss'], 'test_precision': t['precision'], 'test_recall': t['recall'],
            'val_acc_curve': [x['validation_accuracy'] for x in h],
            'val_loss_curve': [x['validation_loss'] for x in h],
            'val_f1_curve': [x['macro_f1'] for x in h],
            'train_loss_curve': [x['training_loss'] for x in h],
            'round_wall_curve': [x['round_time_seconds'] for x in h],
            'round_cpu_curve': [x.get('round_cpu_seconds') for x in h],
            'mean_round_wall_s': float(np.mean([x['round_time_seconds'] for x in h])),
            'mean_round_cpu_s': float(np.mean([x['round_cpu_seconds'] for x in h])),
            'federated_wall_s': d['federated_wall_seconds'], 'federated_cpu_s': d['federated_cpu_seconds'],
            'peak_rss_mb': d['peak_rss_mb'],
            'mean_aggregation_call_wall_s': float(np.mean([x['aggregation_call_wall_seconds'] for x in h])),
            'clients': k,
            # bandwidth per round
            'upload_bytes_per_round': (float(np.mean([x['he_ciphertext_upload_bytes'] for x in h])) if mode == 'ckks'
                                       else k * PT_MODEL_BYTES),
            'download_bytes_per_round': k * PT_MODEL_BYTES,   # global model broadcast is plaintext in both designs
            'server_to_decryptor_bytes_per_round': (float(np.mean([x['he_aggregate_ciphertext_bytes'] for x in h])) if mode == 'ckks' else 0),
            'decryptor_to_server_bytes_per_round': PT_MODEL_BYTES if mode == 'ckks' else 0,
            'one_time_public_context_bytes_per_client': h[0].get('he_public_context_bytes', 0),
        }
        r['total_bytes_per_round'] = (r['upload_bytes_per_round'] + r['download_bytes_per_round'] +
                                      r['server_to_decryptor_bytes_per_round'] + r['decryptor_to_server_bytes_per_round'])
        if mode == 'ckks':
            for key in ('he_packaging_time_seconds', 'he_encryption_time_seconds', 'he_aggregation_time_seconds',
                        'he_decryption_time_seconds', 'he_serialization_time_seconds', 'he_total_time_seconds',
                        'client_encrypt_cpu_seconds', 'aggregation_call_cpu_seconds'):
                r['mean_' + key] = float(np.mean([x[key] for x in h]))
            r['he_ciphertext_count_per_round'] = h[0]['he_ciphertext_count']
            r['per_round_max_abs_err'] = [x['ckks_vs_plaintext_max_abs_err'] for x in h]
            r['per_round_rmse'] = [x['ckks_vs_plaintext_rmse'] for x in h]
            r['max_abs_weight'] = max(x['max_abs_weight'] for x in h)
        summary['runs'][f'{dist}_{mode}'] = r
        pair[mode] = r
    if len(pair) == 2:
        p, c = pair['plaintext'], pair['ckks']
        summary['pairs'][dist] = {
            'delta_test_accuracy_pp': 100 * (c['test_accuracy'] - p['test_accuracy']),
            'delta_test_macro_f1_pp': 100 * (c['test_macro_f1'] - p['test_macro_f1']),
            'delta_test_auroc': c['test_auroc'] - p['test_auroc'],
            'delta_test_loss': c['test_loss'] - p['test_loss'],
            'max_abs_val_acc_diff_pp_any_round': 100 * max(abs(a - b) for a, b in zip(c['val_acc_curve'], p['val_acc_curve'])),
            'added_wall_s_per_round': c['mean_round_wall_s'] - p['mean_round_wall_s'],
            'wall_overhead_pct': 100 * (c['federated_wall_s'] / p['federated_wall_s'] - 1),
            'added_cpu_s_per_round': c['mean_round_cpu_s'] - p['mean_round_cpu_s'],
            'cpu_overhead_pct': 100 * (c['federated_cpu_s'] / p['federated_cpu_s'] - 1),
            'upload_expansion_x': c['upload_bytes_per_round'] / p['upload_bytes_per_round'],
            'total_traffic_expansion_x': c['total_bytes_per_round'] / p['total_bytes_per_round'],
            'extra_peak_rss_mb': c['peak_rss_mb'] - p['peak_rss_mb'],
        }

bench_rows = []
for f in sorted(bench.glob('he_*.json')):
    if f.name == 'he_smoke.json':
        continue
    bench_rows += json.loads(f.read_text())
agg = {}
for r in bench_rows:
    key = f"{r['label']}|{r['params']}|{r['clients']}"
    agg.setdefault(key, []).append(r)
summary['bench'] = {}
for key, rows in agg.items():
    label, params, clients = key.split('|')
    summary['bench'][key] = {'label': label, 'params': params, 'clients': int(clients), 'n': len(rows),
                             **{m: float(np.mean([r[m] for r in rows])) for m in rows[0] if isinstance(rows[0][m], (int, float)) and m not in ('clients', 'repeat')},
                             'ckks_round_wall_s_std': float(np.std([r['ckks_round_wall_s'] for r in rows]))}
out.write_text(json.dumps(summary, indent=1))
print(json.dumps(summary['pairs'], indent=1))


In [ ]:
%%writefile /content/ckks_exp/make_report.py
"""Build a self-contained offline HTML report from analyze.py's summary.json."""
import html, json, math, sys
from pathlib import Path

S = json.loads(Path(sys.argv[1]).read_text())
OUT = Path(sys.argv[2])
runs, pairs, bench = S['runs'], S['pairs'], S.get('bench', {})

# GPU plaintext baseline from the 26 Oct 2026 report (T09 IID / T10 non-IID)
BASE = {'iid': dict(name='T09 (GPU baseline)', acc=.865042, f1=.824045, auroc=.979282, loss=.943519, minutes=22.4),
        'non_iid': dict(name='T10 (GPU baseline)', acc=.861281, f1=.820398, auroc=.979100, loss=.943885, minutes=22.4)}
COL = {'iid_plaintext': '#2563eb', 'iid_ckks': '#93c5fd', 'non_iid_plaintext': '#c2410c', 'non_iid_ckks': '#fdba74'}
LBL = {'iid_plaintext': 'IID · plaintext', 'iid_ckks': 'IID · CKKS', 'non_iid_plaintext': 'Non-IID · plaintext', 'non_iid_ckks': 'Non-IID · CKKS'}
esc = html.escape


def mb(b): return f'{b / 1e6:,.2f} MB'
def pct(x): return f'{100 * x:.2f}%'


def line_chart(series, title, ylab, log=False, w=640, h=300, dash=None, fmt=lambda v: f'{v:.3g}'):
    pad_l, pad_r, pad_t, pad_b = 62, 16, 28, 38
    pts = [v for _, s, _ in series for v in s if v is not None and (not log or v > 0)]
    if not pts:
        return ''
    lo, hi = min(pts), max(pts)
    if log:
        lo, hi = math.floor(math.log10(lo)), math.ceil(math.log10(hi))
        if lo == hi: hi += 1
    elif hi == lo:
        hi = lo + 1
    else:
        span = hi - lo; lo -= .05 * span; hi += .05 * span
    n = max(len(s) for _, s, _ in series)
    X = lambda i: pad_l + (w - pad_l - pad_r) * (i / max(1, n - 1))
    Y = lambda v: pad_t + (h - pad_t - pad_b) * (1 - (((math.log10(v) if log else v) - lo) / (hi - lo)))
    g = [f'<svg viewBox="0 0 {w} {h}" class="chart" role="img" aria-label="{esc(title)}">',
         f'<text x="{pad_l}" y="16" class="ct">{esc(title)}</text>']
    ticks = [10 ** k for k in range(int(lo), int(hi) + 1)] if log else [lo + (hi - lo) * k / 4 for k in range(5)]
    for t in ticks:
        y = Y(t)
        g.append(f'<line x1="{pad_l}" x2="{w - pad_r}" y1="{y:.1f}" y2="{y:.1f}" class="grid"/>'
                 f'<text x="{pad_l - 6}" y="{y + 4:.1f}" class="tick" text-anchor="end">{(f"1e{int(math.log10(t))}" if log else fmt(t))}</text>')
    for i in range(n):
        if n <= 20 and (i == 0 or (i + 1) % 5 == 0):
            g.append(f'<text x="{X(i):.1f}" y="{h - pad_b + 16}" class="tick" text-anchor="middle">{i + 1}</text>')
    g.append(f'<text x="{(w + pad_l) / 2}" y="{h - 6}" class="tick" text-anchor="middle">Communication round</text>'
             f'<text x="14" y="{(h) / 2}" class="tick" transform="rotate(-90 14 {h / 2})" text-anchor="middle">{esc(ylab)}</text>')
    for name, s, color in series:
        d = ' '.join(f'{"M" if j == 0 else "L"}{X(j):.1f},{Y(v):.1f}' for j, v in enumerate(s) if v is not None and (not log or v > 0))
        style = ' stroke-dasharray="6 4"' if dash and name in dash else ''
        g.append(f'<path d="{d}" fill="none" stroke="{color}" stroke-width="2.2"{style}><title>{esc(name)}</title></path>')
        for j, v in enumerate(s):
            if v is not None and (not log or v > 0):
                g.append(f'<circle cx="{X(j):.1f}" cy="{Y(v):.1f}" r="2.6" fill="{color}"><title>{esc(name)} · round {j + 1}: {fmt(v)}</title></circle>')
    g.append('</svg>')
    leg = ''.join(f'<span class="lg"><i style="background:{c}"></i>{esc(nm)}</span>' for nm, _, c in series)
    return f'<figure>{"".join(g)}<figcaption>{leg}</figcaption></figure>'


def hbar(items, title, unit_fmt, w=640):
    rowh = 30; h = 34 + rowh * len(items)
    mx = max(v for _, v, _ in items) or 1
    g = [f'<svg viewBox="0 0 {w} {h}" class="chart" role="img" aria-label="{esc(title)}"><text x="8" y="16" class="ct">{esc(title)}</text>']
    for i, (name, v, c) in enumerate(items):
        y = 28 + i * rowh; bw = (w - 260) * v / mx
        g.append(f'<text x="8" y="{y + 15}" class="tick">{esc(name)}</text><rect x="170" y="{y + 3}" width="{bw:.1f}" height="18" rx="3" fill="{c}"/>'
                 f'<text x="{176 + bw:.1f}" y="{y + 16}" class="tick">{esc(unit_fmt(v))}</text>')
    g.append('</svg>')
    return f'<figure>{"".join(g)}</figure>'


def stacked(rows, title, w=640):
    parts = [('Encode + encrypt (clients)', 'mean_he_encryption_time_seconds', '#7c3aed'),
             ('Serialize / deserialize', 'mean_he_serialization_time_seconds', '#a78bfa'),
             ('Ciphertext addition (server)', 'mean_he_aggregation_time_seconds', '#059669'),
             ('Decrypt + decode (3rd party)', 'mean_he_decryption_time_seconds', '#f59e0b'),
             ('Packing', 'mean_he_packaging_time_seconds', '#94a3b8')]
    rowh = 34; h = 36 + rowh * len(rows)
    mx = max(sum(r[k] for _, k, _ in parts) for _, r in rows) or 1
    g = [f'<svg viewBox="0 0 {w} {h}" class="chart"><text x="8" y="16" class="ct">{esc(title)}</text>']
    for i, (name, r) in enumerate(rows):
        y = 30 + i * rowh; x = 150
        g.append(f'<text x="8" y="{y + 16}" class="tick">{esc(name)}</text>')
        for pn, k, c in parts:
            bw = (w - 230) * r[k] / mx
            g.append(f'<rect x="{x:.1f}" y="{y + 3}" width="{bw:.1f}" height="20" fill="{c}"><title>{esc(pn)}: {r[k]:.3f} s</title></rect>')
            x += bw
        g.append(f'<text x="{x + 6:.1f}" y="{y + 18}" class="tick">{sum(r[k] for _, k, _ in parts):.2f} s</text>')
    g.append('</svg>')
    leg = ''.join(f'<span class="lg"><i style="background:{c};height:10px;width:10px"></i>{esc(pn)}</span>' for pn, _, c in parts)
    return f'<figure>{"".join(g)}<figcaption>{leg}</figcaption></figure>'


# ---------------- content ----------------
cards = []
for dist in ('iid', 'non_iid'):
    if dist in pairs:
        p = pairs[dist]
        cards.append(f'<div class="card"><div class="k">{"IID" if dist == "iid" else "Non-IID"} · test-accuracy change</div><div class="v">{p["delta_test_accuracy_pp"]:+.3f} pp</div><div class="s">CKKS minus plaintext, same seed &amp; partition</div></div>')
if 'iid' in pairs:
    p = pairs['iid']
    cards.append(f'<div class="card"><div class="k">Added wall time per round</div><div class="v">+{p["added_wall_s_per_round"]:.2f} s</div><div class="s">{p["wall_overhead_pct"]:+.1f}% of the federated phase (IID)</div></div>')
    cards.append(f'<div class="card"><div class="k">Upload per client per round</div><div class="v">{p["upload_expansion_x"]:.1f}×</div><div class="s">ciphertext vs float32 model · total traffic {p["total_traffic_expansion_x"]:.1f}×</div></div>')

rows = []
for dist in ('iid', 'non_iid'):
    b = BASE[dist]
    rows.append(f'<tr class="ref"><td>{esc(b["name"])}</td><td>{pct(b["acc"])}</td><td>{pct(b["f1"])}</td><td>{b["auroc"]:.4f}</td><td>{b["loss"]:.3f}</td><td>{b["minutes"]:.1f} min (GPU)</td><td>–</td><td>{mb(5 * 241993 * 4)}</td><td>{mb(2 * 5 * 241993 * 4)}</td></tr>')
    for mode in ('plaintext', 'ckks'):
        k = f'{dist}_{mode}'
        if k in runs:
            r = runs[k]
            rows.append(f'<tr><td><i class="dot" style="background:{COL[k]}"></i>{LBL[k]}</td><td>{pct(r["test_accuracy"])}</td><td>{pct(r["test_macro_f1"])}</td>'
                        f'<td>{r["test_auroc"]:.4f}</td><td>{r["test_loss"]:.3f}</td><td>{r["federated_wall_s"] / 60:.1f} min</td>'
                        f'<td>{r["federated_cpu_s"] / 60:.1f} min</td><td>{mb(r["upload_bytes_per_round"])}</td><td>{mb(r["total_bytes_per_round"])}</td></tr>')

delta_rows = ''.join(
    f'<tr><td>{"IID" if d == "iid" else "Non-IID"}</td><td>{p["delta_test_accuracy_pp"]:+.4f}</td><td>{p["delta_test_macro_f1_pp"]:+.4f}</td>'
    f'<td>{p["delta_test_auroc"]:+.2e}</td><td>{p["delta_test_loss"]:+.2e}</td><td>{p["max_abs_val_acc_diff_pp_any_round"]:.3f}</td>'
    f'<td>+{p["added_wall_s_per_round"]:.2f} s ({p["wall_overhead_pct"]:+.1f}%)</td><td>+{p["added_cpu_s_per_round"]:.2f} s ({p["cpu_overhead_pct"]:+.1f}%)</td>'
    f'<td>{p["upload_expansion_x"]:.1f}×</td><td>{p["total_traffic_expansion_x"]:.1f}×</td><td>{p["extra_peak_rss_mb"]:+.0f} MB</td></tr>'
    for d, p in pairs.items())

acc_chart = line_chart([(LBL[k], [100 * v for v in runs[k]['val_acc_curve']], COL[k]) for k in COL if k in runs],
                       'Global validation accuracy per round', 'Validation accuracy (%)', dash={LBL['iid_ckks'], LBL['non_iid_ckks']}, fmt=lambda v: f'{v:.2f}')
loss_chart = line_chart([(LBL[k], runs[k]['val_loss_curve'], COL[k]) for k in COL if k in runs],
                        'Global validation loss per round', 'Validation loss', dash={LBL['iid_ckks'], LBL['non_iid_ckks']}, fmt=lambda v: f'{v:.3f}')
err_chart = line_chart([(LBL[k], runs[k]['per_round_max_abs_err'], COL[k.replace('ckks', 'plaintext')]) for k in ('iid_ckks', 'non_iid_ckks') if k in runs],
                       'CKKS aggregate vs plaintext FedAvg on the same client models (max |error|)', 'max |w_CKKS − w_plain|', log=True, w=1040, h=280, fmt=lambda v: f'{v:.2e}')
time_chart = line_chart([(LBL[k], runs[k]['round_wall_curve'], COL[k]) for k in COL if k in runs],
                        'Wall-clock time per round', 'Seconds', dash={LBL['iid_ckks'], LBL['non_iid_ckks']}, fmt=lambda v: f'{v:.1f}')
stack = stacked([(LBL[k], runs[k]) for k in ('iid_ckks', 'non_iid_ckks') if k in runs], 'Where the CKKS time goes (mean per round, all 5 clients + server + decryptor)')
bw_items = []
if 'iid_plaintext' in runs and 'iid_ckks' in runs:
    p, c = runs['iid_plaintext'], runs['iid_ckks']
    bw_items = [('Plain: clients→server', p['upload_bytes_per_round'], '#2563eb'),
                ('Plain: server→clients', p['download_bytes_per_round'], '#60a5fa'),
                ('CKKS: clients→server', c['upload_bytes_per_round'], '#7c3aed'),
                ('CKKS: server→decryptor', c['server_to_decryptor_bytes_per_round'], '#a78bfa'),
                ('CKKS: decryptor→server', c['decryptor_to_server_bytes_per_round'], '#c4b5fd'),
                ('CKKS: server→clients', c['download_bytes_per_round'], '#ddd6fe')]
bw_chart = hbar(bw_items, 'Bytes per round by link (5 clients)', mb, w=1040) if bw_items else ''

scal = sorted([b for b in bench.values() if b['label'] == 'client_scaling'], key=lambda b: b['clients'])
scal_rows = ''.join(f'<tr><td>{b["clients"]}</td><td>{b["ckks_round_wall_s"]:.2f} ± {b["ckks_round_wall_s_std"]:.2f} s</td><td>{b["ckks_round_cpu_s"]:.2f} s</td>'
                    f'<td>{b["per_client_encrypt_wall_s"]:.2f} s</td><td>{b["server_plus_ttp_wall_s"]:.2f} s</td><td>{b["plaintext_agg_wall_s"] * 1e3:.1f} ms</td>'
                    f'<td>{mb(b["he_ciphertext_upload_bytes"])}</td><td>{mb(b["plaintext_upload_bytes"])}</td><td>{b["max_abs_err"]:.1e}</td></tr>' for b in scal)
abl = [b for b in bench.values() if b['label'] == 'param_ablation']
abl_rows = ''.join(f'<tr><td>N={b["params"].split(":")[0]}, q={b["params"].split(":")[1]}, Δ=2^{b["params"].split(":")[2]}</td><td>{b["ckks_round_wall_s"]:.2f} s</td>'
                   f'<td>{mb(b["he_ciphertext_upload_bytes"] / 5)}</td><td>{int(b["he_ciphertext_count"] / 5)}</td><td>{b["he_public_context_bytes"] / 1e3:.0f} KB</td><td>{b["max_abs_err"]:.1e}</td></tr>'
                   for b in sorted(abl, key=lambda b: b['he_ciphertext_upload_bytes'], reverse=True))
shm = [b for b in bench.values() if b['label'] == 'ramdisk_serialization']
ser_note = ''
base5 = [b for b in scal if b['clients'] == 5]
if shm and base5:
    ser_note = (f'<p>Serialization uses SEAL\'s file-based <code>save/load</code> through temporary files. Moving those temporary files to RAM (<code>TMPDIR=/dev/shm</code>) changed serialization from '
                f'{base5[0]["he_serialization_time_seconds"]:.2f} s to {shm[0]["he_serialization_time_seconds"]:.2f} s per 5-client round on this Linux machine. '
                'Your earlier Windows/OneDrive measurement (12.3 s of 14.1 s) is therefore mostly a file-system artefact, not a CKKS cost.</p>')

r0 = next(iter(runs.values()))
meta = (f'5 clients · 20 rounds · 5 local epochs · batch 128 · SGD lr 0.01, momentum 0.9 · Dirichlet α = 0.5 for non-IID · seed 42 · '
        f'device: {esc(str(S.get("device", "see runs")))}')

doc = f'''<!DOCTYPE html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>PathMNIST FL — plaintext FedAvg vs CKKS (SEAL) with a third-party decryptor</title>
<style>
:root{{--fg:#0f172a;--mut:#64748b;--bg:#ffffff;--soft:#f1f5f9;--line:#e2e8f0;--acc:#7c3aed}}
@media (prefers-color-scheme:dark){{:root{{--fg:#e2e8f0;--mut:#94a3b8;--bg:#0b1120;--soft:#111a2e;--line:#1e293b}}}}
*{{box-sizing:border-box}}body{{margin:0;font:15px/1.55 system-ui,-apple-system,Segoe UI,Roboto,sans-serif;color:var(--fg);background:var(--bg)}}
main{{max-width:1080px;margin:0 auto;padding:28px 20px 60px}}h1{{font-size:26px;margin:0 0 6px}}h2{{font-size:19px;margin:36px 0 10px;border-top:1px solid var(--line);padding-top:22px}}
.sub{{color:var(--mut)}}.cards{{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:12px;margin:18px 0}}
.card{{background:var(--soft);border-radius:12px;padding:14px 16px}}.k{{color:var(--mut);font-size:13px}}.v{{font-size:26px;font-weight:700;margin:2px 0}}.s{{color:var(--mut);font-size:12.5px}}
table{{border-collapse:collapse;width:100%;font-size:13.5px;margin:8px 0}}th,td{{padding:7px 8px;border-bottom:1px solid var(--line);text-align:right;white-space:nowrap}}th:first-child,td:first-child{{text-align:left}}
th{{color:var(--mut);font-weight:600}}tr.ref td{{color:var(--mut);font-style:italic}}.wrap{{overflow-x:auto}}
.dot{{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:6px}}
.chart{{width:100%;height:auto}}.ct{{font-size:13px;font-weight:600;fill:var(--fg)}}.tick{{font-size:11px;fill:var(--mut)}}.grid{{stroke:var(--line)}}
figure{{margin:14px 0;background:var(--soft);border-radius:12px;padding:12px}}figcaption{{font-size:12.5px;color:var(--mut)}}.lg{{margin-right:14px}}.lg i{{display:inline-block;width:14px;height:3px;vertical-align:middle;margin-right:5px}}
.grid2{{display:grid;grid-template-columns:repeat(auto-fit,minmax(440px,1fr));gap:12px}}.note{{background:var(--soft);border-left:3px solid var(--acc);padding:10px 14px;border-radius:6px}}
code{{background:var(--soft);padding:1px 5px;border-radius:4px}}
</style></head><body><main>
<h1>Plaintext FedAvg vs CKKS-encrypted FedAvg on PathMNIST</h1>
<div class="sub">Microsoft SEAL CKKS (via TenSEAL 0.3.18) · clients encrypt, the server adds ciphertexts with only the public key, an independent third-party decryptor holds the secret key and releases only the averaged model.<br>{meta}</div>
<div class="cards">{"".join(cards)}</div>

<h2>1 · Accuracy: does encryption change the model?</h2>
<div class="wrap"><table><tr><th>Run</th><th>Test acc.</th><th>Macro F1</th><th>AUROC</th><th>Test loss</th><th>FL wall time</th><th>FL CPU time</th><th>Upload / round</th><th>Total traffic / round</th></tr>{"".join(rows)}</table></div>
<p class="sub">Italic rows are the plaintext GPU baseline from the 10-experiment report (same hyperparameters), for reference only: they ran on different hardware, so compare accuracy against them and time only within the paired rows.</p>
<div class="grid2">{acc_chart}{loss_chart}</div>
{err_chart}
<div class="wrap"><table><tr><th>Pair</th><th>Δ test acc (pp)</th><th>Δ macro F1 (pp)</th><th>Δ AUROC</th><th>Δ test loss</th><th>max |Δ val acc| any round (pp)</th><th>Added wall / round</th><th>Added CPU / round</th><th>Upload ×</th><th>Total traffic ×</th><th>Extra peak RAM</th></tr>{delta_rows}</table></div>

<h2>2 · Time and computational cost</h2>
<div class="grid2">{time_chart}{stack}</div>
<h3>Scaling with the number of clients (real 241,993-parameter CNN, one aggregation round)</h3>
<div class="wrap"><table><tr><th>Clients</th><th>CKKS round (wall)</th><th>CKKS round (CPU)</th><th>Encrypt per client</th><th>Server add + decrypt</th><th>Plaintext FedAvg</th><th>Ciphertext upload</th><th>Plaintext upload</th><th>Max error</th></tr>{scal_rows}</table></div>
{ser_note}

<h2>3 · Bandwidth</h2>
{bw_chart}
<p>Each client's model (241,993 values) is packed into 60 ciphertexts of 4,096 slots. A fresh ciphertext holds two polynomials over three 40–60-bit primes, so the serialized upload is ≈19.9 MB per client per round versus 0.97 MB of float32 weights (≈20.5×). The public context (parameters + public key, ≈465 KB) is sent once per client, not every round.</p>
<h3>CKKS parameter ablation (5 clients) — addition-only aggregation needs no multiplicative depth</h3>
<div class="wrap"><table><tr><th>Parameters</th><th>Round time</th><th>Upload per client</th><th>Ciphertexts per client</th><th>Public context</th><th>Max error</th></tr>{abl_rows}</table></div>

<h2>4 · How to read this</h2>
<div class="note"><b>What CKKS protects here:</b> the aggregation server never sees an individual hospital's model, only ciphertexts. It cannot decrypt because the secret key is held by the third-party decryptor, which only receives the <i>sum</i>. <b>What it does not protect:</b> the released global model, a colluding server + decryptor, malicious clients (poisoning), or traffic analysis. Local training is unchanged plaintext PyTorch.</div>
<p class="sub">Timings come from one seed on one machine; the plaintext and CKKS twins ran under identical conditions. Byte counts are serialized SEAL payloads (simulated traffic, no network). The global-model download is counted as plaintext in both designs.</p>
</main></body></html>'''
OUT.write_text(doc, encoding='utf-8')
print('wrote', OUT, len(doc))


### 1 · Sanity check: the repo's CKKS tests (real SEAL encryption) — ~10 s

In [ ]:
!cd /content/FYP && python -m pytest tests -q -m he -p no:cacheprovider 2>&1 | tail -3

### 2 · The four paired training runs (~25 min each on T4)

In [ ]:
import os, subprocess, sys, time
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
RUNS = f'{OUT}/runs'
os.makedirs(RUNS, exist_ok=True)
for dist in ('iid', 'non_iid'):
    for mode in ('plaintext', 'ckks'):
        out = f'{RUNS}/{dist}_{mode}'
        if os.path.exists(f'{out}/federated_history.json'):
            print('skip (done):', out); continue
        if os.path.exists(out):
            import shutil; shutil.rmtree(out)   # incomplete earlier attempt
        t = time.time()
        print(f'=== {dist} / {mode} ===', flush=True)
        r = subprocess.run([sys.executable, '-I', '/content/ckks_exp/run_fl.py', '--repo', '/content/FYP',
                            '--data_dir', '/content/data', '--download', '--out', out, '--mode', mode,
                            '--distribution', dist, '--device', 'auto', '--threads', '2',
                            '--num_clients', '5', '--rounds', '20', '--local_epochs', '5',
                            '--batch_size', '128', '--lr', '0.01', '--seed', '42'],
                           capture_output=True, text=True)
        print(r.stdout[-600:], r.stderr[-2000:] if r.returncode else '', f'({(time.time()-t)/60:.1f} min)', flush=True)

### 3 · HE-only cost benchmark on the real CNN (~3 min, CPU)

In [ ]:
B = f'{OUT}/bench'
os.makedirs(B, exist_ok=True)
S = '/content/ckks_exp/he_cost_bench.py'
!cd /content/FYP && python -I {S} --repo . --out {B}/he_clients.json --label client_scaling --clients 2 3 5 10 --repeats 3 | tail -2
!cd /content/FYP && python -I {S} --repo . --out {B}/he_params.json --label param_ablation --clients 5 --repeats 3 --params "8192:60,40,40,60:40" "8192:60,40,60:40" "16384:60,40,40,60:40" "8192:60,30,60:30" | tail -2
!cd /content/FYP && TMPDIR=/dev/shm python -I {S} --repo . --out {B}/he_shm.json --label ramdisk_serialization --clients 5 --repeats 3 | tail -2

### 4 · Analysis + HTML report

In [ ]:
!python -I /content/ckks_exp/analyze.py {RUNS} {B} {OUT}/summary.json
!python -I /content/ckks_exp/make_report.py {OUT}/summary.json {OUT}/ckks_vs_plaintext_report.html
from IPython.display import HTML, display
display(HTML(open(f'{OUT}/ckks_vs_plaintext_report.html').read()))

In [ ]:
!cd {OUT} && zip -qr /content/results.zip . -x '*.pt'
from google.colab import files
files.download(f'{OUT}/ckks_vs_plaintext_report.html')
files.download('/content/results.zip')